In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
data=pd.read_csv('TRAIN.csv')
data.head()

,F01,F02,F03,F04,F05,F06,F07,F08,F09,F10,...,F39,F40,F41,F42,F43,F44,F45,F46,F47,Class
0,0.185570,0.004568,0.005362,0.003335,0.005415,0.004895,0.012764,0.120138,0.140450,3.361753,...,0.041526,-0.230857,0.003310,0.042250,0.005975,0.002104,0.013878,0.001518,0.011347,0
1,0.369536,0.003983,0.003386,0.004902,0.007570,0.012136,0.118050,0.323925,0.132093,2.766117,...,-0.141285,-6.222857,0.834177,0.227968,0.018463,-0.020487,0.001246,0.007489,0.008724,1
2,0.602510,0.008442,0.012961,0.012870,0.046885,0.115401,0.065688,0.306677,0.498805,4.521201,...,0.011334,10.335251,-0.276614,-0.198900,-0.012756,0.014286,-0.001866,0.002687,0.013452,1
3,0.347957,0.064721,0.013611,0.011541,0.006492,0.008690,0.013192,0.164553,0.298665,3.170847,...,0.190479,2.864912,-1.921939,0.891690,1.108098,0.635431,0.081368,-0.000225,0.009166,0
4,0.233653,0.012217,0.010088,0.022095,0.026040,0.015062,0.016063,0.084648,0.213367,8.150943,...,0.203164,0.001812,-0.092731,0.005280,-0.213985,0.032195,0.002081,0.028930,-0.025912,1


In [25]:
X=data.drop('Class',axis=1)
y=data['Class']

In [4]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.25,random_state=42)

In [7]:
!conda install -c conda-forge xgboost -y

3 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - conda-forge
 - defaults
Platform: win-64



CondaHTTPError: HTTP 000 CONNECTION FAILED for url <https://conda.anaconda.org/conda-forge/win-64/repodata.json>
Elapsed: -

An HTTP error occurred when trying to retrieve this URL.
HTTP errors are often intermittent, and a simple retry will get you on your way.
'https//conda.anaconda.org/conda-forge/win-64'




In [16]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score,accuracy_score,confusion_matrix,classification_report

In [10]:
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=42
)

xgb.fit(X_train, y_train)

y_prob = xgb.predict_proba(X_test)[:,1]

print("AUC:", roc_auc_score(y_test, y_prob))

AUC: 0.9966185527776228


In [11]:
y_pred=xgb.predict(X_test)

In [17]:
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.9740801656920078
[[19585   215]
 [  636 12396]]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     19800
           1       0.98      0.95      0.97     13032

    accuracy                           0.97     32832
   macro avg       0.98      0.97      0.97     32832
weighted avg       0.97      0.97      0.97     32832



In [18]:
# lets try hyper parameter tuning
param_dist={
    "max_depth":[4,6,8],
    "learning_rate":[0.01,0.05,0.1],
    "n_estimators":[300,500,800],
    "subsample":[0.7,0.8,0.9],
    "colsample_bytree":[0.6,0.8,1]
}

In [19]:
from sklearn.model_selection import RandomizedSearchCV
random_search=RandomizedSearchCV(
    xgb,
    param_dist,
    n_iter=25,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    verbose=1
    )

In [ ]:
random_search.fit(X_train,y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=0.8, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric='auc',
                                           feature_types=None,
                                           feature_weights=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_constrain...
                                           max_leaves=None,
                                           min_child_weight=None, missing=nan,
                                           monotone_constraints=None,
                                           multi_strategy=None,
                                           n_estimators=300, n_jobs=None,
                                           num_parallel_tree=None, ...),
                   n_iter=25, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.6, 0.8, 1],
                                        'learning_rate': [0.01, 0.05, 0.1],
                                        'max_depth': [4, 6, 8],
                                        'n_estimators': [300, 500, 800],
                                        'subsample': [0.7, 0.8, 0.9]},
                   scoring='roc_auc', verbose=1)

In [21]:
y_pred_cv=random_search.predict(X_test)

In [22]:
y_prob_cv=random_search.predict_proba(X_test)[:,1]

In [26]:
print(accuracy_score(y_test,y_pred_cv))
print(confusion_matrix(y_test,y_pred_cv))
print(classification_report(y_test,y_pred_cv))
print("AUC:", roc_auc_score(y_test, y_prob_cv))

0.9763036062378168
[[19592   208]
 [  570 12462]]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     19800
           1       0.98      0.96      0.97     13032

    accuracy                           0.98     32832
   macro avg       0.98      0.97      0.98     32832
weighted avg       0.98      0.98      0.98     32832

AUC: 0.9970483417663437


In [24]:
#Cross Validating our model
from sklearn.model_selection import cross_val_score

score=cross_val_score(xgb,X_train,y_train,cv=5,scoring='roc_auc')

print(score)
print(score.mean())

[0.99629463 0.99653915 0.99301081 0.9952938  0.99643946]
0.9955155705634067
